In [2]:
from wildlife_datasets.datasets import MacaqueFaces
from wildlife_tools.data import WildlifeDataset
import torchvision.transforms as T

metadata = MacaqueFaces('datasets/MacaqueFaces')
transform = T.Compose([T.Resize([224, 224]), T.ToTensor(), T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))])
dataset = WildlifeDataset(metadata.df, metadata.root, transform=transform)

In [3]:
dataset_database = WildlifeDataset(metadata.df.iloc[100:,:], metadata.root, transform=transform)
dataset_query = WildlifeDataset(metadata.df.iloc[:100,:], metadata.root, transform=transform)

In [5]:
import timm
from wildlife_tools.features import DeepFeatures

name = 'hf-hub:BVRA/MegaDescriptor-T-224'
extractor = DeepFeatures(timm.create_model(name, num_classes=0, pretrained=True), num_workers=0)
query, database = extractor(dataset_query), extractor(dataset_database)

100%|███████████████████████████████████████████████████████████████| 49/49 [05:13<00:00,  6.41s/it]


In [7]:
from wildlife_tools.similarity import CosineSimilarity

similarity_function = CosineSimilarity()
similarity = similarity_function(query, database)

In [ ]:
import numpy as np
from wildlife_tools.inference import KnnClassifier
classifier = KnnClassifier(
    k=1,
    database_labels=dataset_database.labels_string
)

predictions = classifier(similarity["cosine"])

accuracy = np.mean(
    dataset_query.labels_string == predictions
)

print(f"Accuracy: {accuracy:.4f}")

Accuracy: 1.0000


C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\wildlife_tools\inference\classifier.py:61: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  results = pd.DataFrame(results).T.fillna(method="ffill").T


## Testovanie natrenovaneho

In [11]:
import torch
import timm
import numpy as np

from wildlife_tools.features import DeepFeatures
from wildlife_tools.similarity import CosineSimilarity
from wildlife_tools.inference import KnnClassifier

# ---------------------------
# Load fine-tuned model
# ---------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = timm.create_model(
    "hf-hub:BVRA/MegaDescriptor-T-224",
    pretrained=False,
    num_classes=0
)
checkpoint = torch.load(
    "trained_models/megadescriptor_t224_lynx.pth",
    map_location=device,
    weights_only=False
)
model.load_state_dict(checkpoint["model"])
model = model.to(device)
model.eval()

print("Loaded fine-tuned model")

# ---------------------------
# Extract embeddings
# ---------------------------
extractor = DeepFeatures(
    model,
    batch_size=16,
    device=str(device),
    num_workers=0
)

print("Extracting query embeddings...")
query_embeddings = extractor(dataset_query)

print("Extracting database embeddings...")
database_embeddings = extractor(dataset_database)

print("Query shape:", query_embeddings.shape)
print("Database shape:", database_embeddings.shape)

# ---------------------------
# Similarity
# ---------------------------
similarity = CosineSimilarity()(
    query_embeddings,
    database_embeddings
)

# ---------------------------
# KNN classification
# ---------------------------
classifier = KnnClassifier(
    k=1,
    database_labels=dataset_database.labels_string
)

predictions = classifier(similarity["cosine"])

# ---------------------------
# Accuracy
# ---------------------------
accuracy = np.mean(
    dataset_query.labels_string == predictions
)

print(f"Fine-tuned MegaDescriptor accuracy: {accuracy:.4f}")

Loaded fine-tuned model
Extracting query embeddings...


100%|█████████████████████████████████████████████████████████████████| 7/7 [00:01<00:00,  6.20it/s]


Extracting database embeddings...


100%|█████████████████████████████████████████████████████████████| 387/387 [00:45<00:00,  8.48it/s]

Query shape: (100, 768)
Database shape: (6180, 768)
Fine-tuned MegaDescriptor accuracy: 1.0000



C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\wildlife_tools\inference\classifier.py:61: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  results = pd.DataFrame(results).T.fillna(method="ffill").T


In [12]:
len(set(dataset_query.labels_string) - set(dataset_database.labels_string))

0

In [14]:
print("Overlapping identities:",
      len(set(dataset_database.labels_string)
          & set(dataset_query.labels_string)))

print("Database size:", len(dataset_database.labels_string))
print("Query size:", len(dataset_query.labels_string))

print("Unique DB identities:", len(set(dataset_database.labels_string)))
print("Unique Q identities:", len(set(dataset_query.labels_string)))

Overlapping identities: 1
Database size: 6180
Query size: 100
Unique DB identities: 34
Unique Q identities: 1
